# Full GSM8K perturbation — vanilla AND SIM-CoT, on A100

Runs the perturbation_divergence experiments on the **full** GSM8K test+train
split (N=8,792) for **both training paradigms** at σ ∈ {0.1, 1.0}:

| paradigm | method  | sigma | output suffix       |
|---|---|---:|---|
| vanilla  | Coconut | 1.0   | `_sigma1.0_full`    |
| vanilla  | CODI    | 1.0   | `_sigma1.0_full`    |
| vanilla  | Coconut | 0.1   | `_sigma0.1_full`    |
| vanilla  | CODI    | 0.1   | `_sigma0.1_full`    |
| SIM-CoT  | Coconut | 1.0   | `_sigma1.0_full`    |
| SIM-CoT  | CODI    | 1.0   | `_sigma1.0_full`    |
| SIM-CoT  | Coconut | 0.1   | `_sigma0.1_full`    |
| SIM-CoT  | CODI    | 0.1   | `_sigma0.1_full`    |

## No-overwrite contract

Every output filename and every stability.h5 key gets the suffix
`_sigma{σ}_full` so it cannot collide with what is already in the repo:

- The **vanilla σ=0.01 (N=500)** plots from the original `runner.py` (no suffix)
  and the **vanilla σ ∈ {0.1, 1.0} N=500 stratified** plots Shreya pushed earlier
  today (`_sigma0.1`, `_sigma1.0` without `_full`) **stay intact**.
- The **SIM-CoT σ=0.01 (N=500)** plots in `results/simcot_*_literal/` from
  yesterday's SIM-CoT perturbation hand-off (no suffix) **stay intact**.
- This notebook only writes new files with the `_full` token in the name.

## Estimated runtime

| Block | A100 ETA | Cumulative |
|---|---:|---:|
| Setup (clone, deps, GPU, checkpoints, JSON) | 10 min | 0:10 |
| Bootstrap SIM-CoT release → translated ckpts | 5 min | 0:15 |
| Smoke test (N=10, σ=1.0) | < 1 min | 0:15 |
| Vanilla Coconut σ=1.0 (full 8792) | ~40 min | 0:55 |
| Vanilla CODI σ=1.0 (full 8792)    | ~80 min | 2:15 |
| Vanilla Coconut σ=0.1 (full 8792) | ~40 min | 2:55 |
| Vanilla CODI σ=0.1 (full 8792)    | ~80 min | 4:15 |
| SIM-CoT Coconut σ=1.0 (full 8792) | ~40 min | 4:55 |
| SIM-CoT CODI σ=1.0 (full 8792)    | ~80 min | 6:15 |
| SIM-CoT Coconut σ=0.1 (full 8792) | ~40 min | 6:55 |
| SIM-CoT CODI σ=0.1 (full 8792)    | ~80 min | 8:15 |
| Bundle + Drive sync | < 5 min | 8:20 |

**Total ≈ 8 hours.** Outputs sync to Drive at the end so a Colab tab disconnect
does not lose work; the kernel keeps running.

Author: Shreya → hand-off to Manju (A100)

## 1. Environment setup — mount Drive, clone repo, checkout branch

In [ ]:
from pathlib import Path
import os, sys

from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = Path('/content/drive/MyDrive/perturbation_full_gsm8k_outputs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print('Outputs sync to:', DRIVE_OUT)

REPO = Path('/content/Latent-COT')
if not REPO.is_dir():
    !git clone https://github.com/SabariIyyappan/Latent-CoT.git {REPO}
%cd {REPO}
!git fetch origin
!git checkout exp-simcot-gsm8k
!git pull origin exp-simcot-gsm8k
!git log --oneline -5

## 2. Install pinned dependencies

`transformers==4.52.4` is required for the `cache_position` patch in
`analysis/wrappers.py` to land where it should.

In [ ]:
!pip uninstall -q -y torchao 2>&1 | tail -2
!pip install -q --upgrade pip
!pip install -q \
    transformers==4.52.4 \
    accelerate \
    peft \
    huggingface_hub \
    'h5py>=3.11' \
    datasets \
    pydmd \
    phate \
    umap-learn \
    scikit-learn \
    matplotlib \
    numpy 2>&1 | tail -8
print('install done')

## 3. GPU check (must be A100 — abort if not)

In [ ]:
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Set Runtime → Change runtime type → GPU → A100')
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'device: {name}   VRAM: {vram:.1f} GB')
if 'A100' not in name:
    print(f'WARNING: {name} is slower than A100. Estimates above will be roughly:')
    print('  T4: 5x   V100: 2-3x   L4: 2x')

## 4. Stage all four checkpoints

Two paradigms × two methods = 4 checkpoints we need on disk:

| repo path | source | role |
|---|---|---|
| `checkpoints/coconut/`             | `ModalityDance/latent-tts-coconut` (HF) | vanilla Coconut |
| `checkpoints/codi/`                | `ModalityDance/latent-tts-codi`    (HF) | vanilla CODI    |
| `checkpoints/simcot_coconut_gpt2/` | `internlm/SIM_COT-GPT2-Coconut` (HF) → bootstrap-translate | SIM-CoT Coconut |
| `checkpoints/simcot_codi_gpt2/`    | `internlm/SIM_COT-GPT2-CODI`    (HF) → bootstrap-translate | SIM-CoT CODI    |

We pull from HuggingFace because the repo's GitHub LFS budget is exhausted.
The bootstrap step translates upstream's release format into our
`LatentGenerationMixin`-compatible layout (same procedure as Path A in
`scripts/finetune/bootstrap_simcot_*.py`).

In [ ]:
# 4.1 Vanilla checkpoints (direct snapshot_download — no transformation needed)
from huggingface_hub import snapshot_download

CKPTS = {
    'coconut': ('ModalityDance/latent-tts-coconut', REPO / 'checkpoints' / 'coconut'),
    'codi':    ('ModalityDance/latent-tts-codi',    REPO / 'checkpoints' / 'codi'),
}
for label, (repo_id, local) in CKPTS.items():
    if not (local / 'config.json').exists():
        snapshot_download(repo_id=repo_id, local_dir=str(local), local_dir_use_symlinks=False)
    n_files = len(list(local.iterdir()))
    print(f'  {label:<8s} {local}  ({n_files} files)')

In [ ]:
# 4.2 Bootstrap SIM-CoT checkpoints (pulls upstream + reformats)
%cd {REPO}
for tgt in ['checkpoints/simcot_coconut_gpt2', 'checkpoints/simcot_codi_gpt2']:
    if (REPO / tgt / 'config.json').exists():
        print(f'  exists: {tgt}  (skip)')
    else:
        bootstrap = ('bootstrap_simcot_coconut.py' if 'coconut' in tgt
                     else 'bootstrap_simcot_codi.py')
        print(f'  bootstrapping {tgt}...')
        !python scripts/finetune/{bootstrap} --out {tgt} 2>&1 | tail -5

for tgt in ['checkpoints/simcot_coconut_gpt2', 'checkpoints/simcot_codi_gpt2']:
    n_files = len(list((REPO / tgt).iterdir()))
    print(f'  {tgt}  ({n_files} files)')

## 5. Pre-stage the GSM8K question JSON

8,792 rows = full test + train. Built once, reused by every run.

In [ ]:
import json
JSON_PATH = REPO / 'data' / 'gsm8k_for_literal' / 'gsm8k_train_test.json'
if not JSON_PATH.exists():
    JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    from datasets import load_dataset
    rows = []
    for split, sp in [('train', 'train'), ('test', 'test')]:
        for r in load_dataset('gsm8k', 'main', split=sp):
            rows.append({'question': r['question'], 'answer': r['answer'], '_split': split})
    with JSON_PATH.open('w', encoding='utf-8') as f:
        json.dump(rows, f)
    print(f'Wrote {len(rows)} rows to {JSON_PATH}')
else:
    print(f'Already present: {JSON_PATH}')
with JSON_PATH.open(encoding='utf-8') as f:
    data = json.load(f)
print(f'Total rows: {len(data)}    first: {data[0]["question"][:80]}')

## 6. Smoke test — N=10 on vanilla Coconut at σ=1.0

Sanity check before launching the multi-hour runs. Should finish in well under a
minute on A100. If anything looks off, fix it BEFORE the long runs — they're long
enough that a silent failure halfway is a real cost.

Smoke artefacts use suffix `_smoke` so they're unambiguous and the cleanup cell
removes them before we proceed.

In [ ]:
%cd {REPO}
!PYTHONIOENCODING=utf-8 python -u scripts/add_perturbation.py \
    --method coconut --checkpoint checkpoints/coconut \
    --out_dir results/coconut_gpt2/20260502_191252 \
    --questions_json data/gsm8k_for_literal/gsm8k_train_test.json \
    --n_samples 10 --noise_std 1.0 \
    --filename_suffix _smoke --stability_key_suffix _smoke 2>&1 | tail -20

In [ ]:
# Clean smoke artefacts
import h5py
smoke_plot = REPO / 'results/coconut_gpt2/20260502_191252/plots/perturbation_divergence_smoke.png'
if smoke_plot.exists():
    smoke_plot.unlink()
stab = REPO / 'results/coconut_gpt2/20260502_191252/stability_feats/stability.h5'
if stab.exists():
    with h5py.File(str(stab), 'a') as f:
        for k in list(f.keys()):
            if '_smoke' in k:
                del f[k]
print('smoke artefacts cleared')

## 7. Configuration matrix — single source of truth for every downstream cell

Defining `RUNS` once and looping over it everywhere (run, inspect, bundle)
removes per-run code duplication. Edit this list to add or skip combinations.

In [ ]:
# Each entry is a fully self-describing run.
# tag = used in commit messages / bundle layout / log filenames.
# checkpoint and out_dir are paths relative to REPO root.
RUNS = [
    # paradigm , method  , checkpoint                          , out_dir                                    , sigma
    ('vanilla' , 'coconut', 'checkpoints/coconut'             , 'results/coconut_gpt2/20260502_191252'    , 1.0),
    ('vanilla' , 'codi'   , 'checkpoints/codi'                , 'results/codi_gpt2/20260502_191252'       , 1.0),
    ('vanilla' , 'coconut', 'checkpoints/coconut'             , 'results/coconut_gpt2/20260502_191252'    , 0.1),
    ('vanilla' , 'codi'   , 'checkpoints/codi'                , 'results/codi_gpt2/20260502_191252'       , 0.1),
    ('simcot'  , 'coconut', 'checkpoints/simcot_coconut_gpt2' , 'results/simcot_coconut_literal'          , 1.0),
    ('simcot'  , 'codi'   , 'checkpoints/simcot_codi_gpt2'    , 'results/simcot_codi_literal'             , 1.0),
    ('simcot'  , 'coconut', 'checkpoints/simcot_coconut_gpt2' , 'results/simcot_coconut_literal'          , 0.1),
    ('simcot'  , 'codi'   , 'checkpoints/simcot_codi_gpt2'    , 'results/simcot_codi_literal'             , 0.1),
]

def suffix_for(sigma):
    return f'_sigma{sigma}_full'

def tag_for(paradigm, method, sigma):
    return f'{paradigm}_{method}_sigma{sigma}'

print(f'Configured {len(RUNS)} runs:')
for paradigm, method, ckpt, out, sigma in RUNS:
    print(f'  {paradigm:>7s}  {method:>7s}  σ={sigma:<3}  {ckpt:<40s}  -> {out}')

## 8. Run every entry in the matrix sequentially

Each run writes:
- `<out_dir>/plots/perturbation_divergence{suffix}.png`
- merges keys with `{suffix}` into `<out_dir>/stability_feats/stability.h5`
- `logs/perturb_{tag}.log`

Outputs from one run never collide with outputs from another (suffix differs)
and never collide with prior σ=0.01 / N=500 results (the `_full` token in the
suffix is unique to this notebook).

In [ ]:
%cd {REPO}
import time, subprocess, sys
LOG_DIR = REPO / 'logs'; LOG_DIR.mkdir(exist_ok=True)

summary = []
for i, (paradigm, method, ckpt, out, sigma) in enumerate(RUNS, start=1):
    tag = tag_for(paradigm, method, sigma)
    suffix = suffix_for(sigma)
    log = LOG_DIR / f'perturb_{tag}.log'
    print(f'\n{"="*72}\nRun {i}/{len(RUNS)}: {tag.upper()}\n{"="*72}')
    cmd = [
        sys.executable, '-u', 'scripts/add_perturbation.py',
        '--method', method,
        '--checkpoint', ckpt,
        '--out_dir', out,
        '--questions_json', 'data/gsm8k_for_literal/gsm8k_train_test.json',
        '--n_samples', '0',                       # 0 = full N=8792
        '--noise_std', str(sigma),
        '--filename_suffix', suffix,
        '--stability_key_suffix', suffix,
    ]
    t0 = time.time()
    with log.open('w') as fl:
        rc = subprocess.run(
            cmd, stdout=fl, stderr=subprocess.STDOUT,
            env={**os.environ, 'PYTHONIOENCODING': 'utf-8'},
        ).returncode
    elapsed = (time.time() - t0) / 60
    summary.append((tag, rc, elapsed))
    print(f'  finished in {elapsed:.1f} min   rc={rc}')
    !tail -8 {log}

print('\n' + '=' * 72 + '\nALL RUNS DONE\n' + '=' * 72)
for tag, rc, elapsed in summary:
    status = 'OK' if rc == 0 else f'FAILED rc={rc}'
    print(f'  {tag:>30s} : {elapsed:>6.1f} min   {status}')

## 9. Inspect outputs — loops over the same RUNS matrix

Verifies every expected PNG landed and renders each inline.

In [ ]:
from IPython.display import Image, display, Markdown

missing = []
for paradigm, method, ckpt, out, sigma in RUNS:
    suffix = suffix_for(sigma)
    p = REPO / out / 'plots' / f'perturbation_divergence{suffix}.png'
    label = f'{paradigm} {method.upper()} σ={sigma}'
    if p.exists():
        display(Markdown(f'### {label}'))
        display(Image(str(p)))
    else:
        missing.append(str(p))
        print(f'MISSING: {p}')

if not missing:
    print(f'\n✅ All {len(RUNS)} expected plots present.')
else:
    print(f'\n⚠️  {len(missing)} plot(s) missing. Inspect logs/ before bundling.')

## 10. Bundle for hand-off — Drive sync

Copies every `_full` plot and HDF5 key into a single zip + Drive copy. The
bundle layout mirrors the RUNS matrix so the local pickup is mechanical.

In [ ]:
import shutil, h5py
from datetime import datetime

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
bundle_root = REPO / f'perturbation_full_gsm8k_{stamp}'
bundle_root.mkdir(exist_ok=True)

# Group by (paradigm, method, out_dir) — each is one tree we touch
trees = {}
for paradigm, method, ckpt, out, sigma in RUNS:
    trees.setdefault((paradigm, method, out), []).append(sigma)

for (paradigm, method, out), sigmas in trees.items():
    folder = bundle_root / f'{paradigm}_{method}'
    folder.mkdir(exist_ok=True)
    src_plots = REPO / out / 'plots'
    src_stab  = REPO / out / 'stability_feats' / 'stability.h5'
    # Copy the _full PNGs
    for sigma in sigmas:
        suffix = suffix_for(sigma)
        p = src_plots / f'perturbation_divergence{suffix}.png'
        if p.exists():
            shutil.copy2(p, folder / p.name)
    # Extract just the _full keys from the per-tree stability.h5
    if src_stab.exists():
        dst = folder / 'perturbation_full_keys.h5'
        with h5py.File(str(src_stab), 'r') as fin, h5py.File(str(dst), 'w') as fout:
            for k in fin.keys():
                if '_full' in k:
                    fout.create_dataset(k, data=fin[k][:])
        if not list(h5py.File(str(dst), 'r').keys()):
            dst.unlink()

# Manifest
manifest_lines = [
    'Full GSM8K perturbation results (N=8792)',
    '=========================================',
    f'Generated: {stamp}',
    f'Configured runs: {len(RUNS)}',
    'noise_std values: 0.1, 1.0',
    'n_perturbations per question: 3',
    '',
    'Folder layout:',
]
for (paradigm, method, out), sigmas in trees.items():
    sigs = ', '.join(f'σ={s}' for s in sigmas)
    manifest_lines.append(f'  {paradigm}_{method}/  ({sigs})  source={out}')
manifest_lines += [
    '',
    'No-overwrite contract:',
    '  - Existing plots / HDF5 keys without the _full token are untouched.',
    '  - This bundle only ships *_full artefacts.',
    '',
    'Source code: scripts/add_perturbation.py @ exp-simcot-gsm8k',
    'Notebook: notebooks/perturbation_full_gsm8k_colab.ipynb',
]
(bundle_root / 'README.txt').write_text('\n'.join(manifest_lines))

shutil.make_archive(str(bundle_root), 'zip', root_dir=bundle_root.parent, base_dir=bundle_root.name)
zip_path = Path(str(bundle_root) + '.zip')
drive_zip = DRIVE_OUT / zip_path.name
shutil.copy2(zip_path, drive_zip)
print(f'Bundle:           {zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)')
print(f'Synced to Drive:  {drive_zip}')
print('\nSend the Drive path back to Shreya for local pickup.')

## 11. (Optional) Per-step mean summary table

Tabulates mean divergence at each latent step `t = 0..5` per run, formatted for
paste-in to the results section.

In [ ]:
import h5py, numpy as np

header = f'{"paradigm":>8s}  {"method":>7s}  {"σ":>4s}  step-mean  t=0..5'
print(header)
print('-' * len(header))
for paradigm, method, ckpt, out, sigma in RUNS:
    suffix = suffix_for(sigma)
    stab = REPO / out / 'stability_feats' / 'stability.h5'
    if not stab.exists():
        print(f'{paradigm:>8s}  {method:>7s}  {sigma:>4}  STAB FILE MISSING')
        continue
    with h5py.File(str(stab), 'r') as f:
        key = f'perturbation_divergence{suffix}'
        if key not in f:
            print(f'{paradigm:>8s}  {method:>7s}  {sigma:>4}  KEY MISSING ({key})')
            continue
        sm = np.nanmean(f[key][:], axis=0)
    print(f'{paradigm:>8s}  {method:>7s}  {sigma:>4}  ' + ' '.join(f'{v:7.2f}' for v in sm))